# Survey Answer Clustering and Visualization
This notebook processes open-ended survey answers as follows (WIP -- we might update things):
1. Load TSV and filter empty 'target' rows.
2. Compute embeddings with Qwen/Qwen3-Embedding-8B.
3. Cluster with HDBScan (max 6 clusters).
4. Extract top 5 n-grams per cluster.
5. Visualize clusters with UMAP and overlay frequent n-grams.

**Note: to run this notebook with a Qwen embedder, you need a cuda device.**

## Imports and Utils

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Import required libraries
import os
import pandas as pd
import numpy as np
import random
from sentence_transformers import SentenceTransformer
# import hdbscan
import umap
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from collections import Counter
import seaborn as sns
import sklearn
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.representation import LiteLLM
from hdbscan import HDBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize, StandardScaler
import torch
from bertopic.backend import BaseEmbedder

from dotenv import load_dotenv
load_dotenv("../.env")

In [ ]:
# Load TSV and filter empty 'target' rows
tsv_path = "../survey_v3_clean.tsv"
main_df = pd.read_csv(tsv_path, sep='\t', skiprows=[1], encoding='utf-8')
print(main_df.columns)

In [ ]:
def prepare_texts(main_df: pd.DataFrame, target_col: str):
    df = main_df.copy()

    df = df[df[target_col].notnull() & (df[target_col].str.strip() != '')]
    target_texts = df[target_col]

    # minor processing
    target_texts = target_texts.apply(lambda x: x.lower().strip())

    print(f"Number of rows with a value in column {target_col}: {len(target_texts)} ({len(target_texts) / len(main_df) * 100:.2f}%)")
    print(f"First two examples: {target_texts.values[:2]}")
    return target_texts.tolist()

In [ ]:
open_text_Qs = {
    "Q46": "Ci sono ragioni per cui scegli di usare una specifica lingua rispetto ad altre?",
    "Q48": "Ci sono ragioni specifiche per cui preferisci scrivere o parlare?",
    "Q85": "Vuoi raccontarci della tua esperienza e degli errori o stereotipi riscontrati?",
    "strategies_5_TEXT": "Quando utilizzi queste tecnologie e assistenti di dialogo, quali strategie usi per ottenere risposte più precise o utili?"
}

In [ ]:
summarization_prompt = """\
I wrote and published a survey to understand how Italians use generative AI. The survey and the participant responses are in Italian.
Now, I am analyzing the following open-form question:

# Question
<<question>>

After clustering my answers, I have identified a topic described by the following keywords. 

# Keywords
[KEYWORDS]

The following responses are a representative sample of the cluster representing the topic

# Documents
[DOCUMENTS]

# Guidelines
Based on this information, your task it to generate a topic description in the context of this question. Here are the guidelines:
1. The description must be detailed and exhaustive, but it CANNOT exceed 300 words.
2. Describe what brought you to the description by mentioning the sample answers. 
3. Do not mention the question, it's known to the reader. Moreover, do not make any assumption of the number of responses in the cluster.
4. Begin the description with a short title that summarizes it, e.g., "Enhanced perceived precision. <description>", or "Chosen due to convenience. <description>"
5. If the answers are simply "no" or "nessuna", this is a cluster with negative answers. In this case, just avoid spending many words and say "Negative answers". 

Begin the answer right after this prompt.
"""

In [ ]:
def apply_PCA(X):
    # 0) Optional: scale or not — for text embeddings unit-normalization often suffices
    Xn = normalize(X, norm='l2', axis=1)   # recommended for cosine-structure
    
    # 1) PCA to denoise / speed up (only if d_emb is large)
    # choose n_pca ~ min(300, d_emb) or less (50-200 often works well)
    n_pca = 200
    if Xn.shape[1] > n_pca:
        pca = PCA(n_components=n_pca, random_state=42)
        Xr = pca.fit_transform(Xn)
    else:
        Xr = Xn
    return Xr, pca

def parse_topic_info(topic_info):
    print(f"Num topics: {len(topic_info)}")
    topic_info = topic_info.drop(columns=["Name"])
    topic_info["Representation"] = topic_info["Representation"].apply(lambda x: x[0])
    return topic_info

class TopicModel:
    def __init__(self, embed_model: str = "base"):
        self.embed_model = embed_model
        self._embeds = None
        self._topic_model = None
        if embed_model == "qwen":
            # Then we can precompute embeddings with a better model
            print("Loading Qwen3 embedder...")
            self.embedder = SentenceTransformer(
                "Qwen/Qwen3-Embedding-8B",
                model_kwargs={"device_map": "auto"},
                tokenizer_kwargs={"padding_side": "left"},
            )
        elif embed_model == "jina":
            self.embedder = SentenceTransformer("jinaai/jina-embeddings-v4", trust_remote_code=True)

    def compute_or_load_embeds(self, texts: list[str], reuse_embeddings: bool):
        if self._embeds is None or not reuse_embeddings:
            print(f"Embedding the texts (better if on CUDA)...")
            embeddings = self.embedder.encode(texts, show_progress_bar=True, device="cuda")
            self._embeds = embeddings
        return self._embeds

    def compute_topics(
        self,
        question: str,
        texts: list[str],
        summarization_model: str = "openrouter/google/gemma-3-27b-it:free",
        nr_docs: int = 10,
        diversity: float = 0.1,
        umap_n_components: int = 5,
        umap_n_neighbors: int = 15,
        hdbscan_metric: str = "euclidean",
        hdbscan_min_cluster_size: int = 15,
        hdbscan_min_samples: int = None,
        hdbscan_cluster_selection_method: str = "eom",
        exclude_stopwords: list[str] = [],
        reuse_embeddings: bool = True
    ):  
        # 1. Dimensionality reduction
        # representation_model = KeyBERTInspired()
        umap_model = umap.UMAP(
            n_neighbors=umap_n_neighbors,
            n_components=umap_n_components,
            min_dist=0.0,
            metric='cosine',
            random_state=42
        )

        # Stopwords and tokenization
        with open("stopwords_it.txt") as fp:
            swds = [f.strip() for f in fp.readlines() if f.strip() not in exclude_stopwords]
        vectorizer_model = CountVectorizer(stop_words=swds, ngram_range=(1, 3))
        
        # 2. Clustering
        hdbscan_model = HDBSCAN(
            min_cluster_size=hdbscan_min_cluster_size,
            metric=hdbscan_metric,
            prediction_data=True,
            min_samples=hdbscan_min_samples,
            cluster_selection_method=hdbscan_cluster_selection_method
        )

        # 3. Summarization model to interpret clusters
        if summarization_model == "keybert":
             representation_model = KeyBERTInspired()
        else:
            representation_model = LiteLLM(
                prompt=summarization_prompt.replace("<<question>>", question),
                model=summarization_model,
                nr_docs=nr_docs,
            )
    
        kwargs = {
            "vectorizer_model": vectorizer_model,
            "umap_model": umap_model,
            "representation_model": representation_model,
            "hdbscan_model": hdbscan_model,
            "top_n_words": 10,
        }
        input_args = (texts, )

        if self.embed_model == "base":
            kwargs["language"] = "multilingual"    
        elif self.embed_model in ["qwen", "jina"]:
            embeddings = self.compute_or_load_embeds(texts, reuse_embeddings)
            
            # Optionally do PCA before topic extraction to reduce input dimensionality
            # embeddings, pca_model = apply_PCA(embeddings)

            # Inputs are the embeddings instead of raw texts in this case.
            input_args += (embeddings,)
            kwargs["embedding_model"] = self.embedder
            
        topic_model = BERTopic(
            **kwargs
        )
        self._topic_model = topic_model
    
        print("Computing topics...")
        topics, probs = topic_model.fit_transform(*input_args)
        topic_info = parse_topic_info(topic_model.get_topic_info())

        hier_topics = topic_model.hierarchical_topics(texts)
        return topic_info, topics, hier_topics

    def merge_topics(self, texts, ids_list):
        self._topic_model.merge_topics(texts, ids_list)
        topic_info = parse_topic_info(self._topic_model.get_topic_info())
        topics = self._topic_model.topics_
        return topic_info, topics

    def reduce_outliers(self, texts, topics):
        new_topics = self._topic_model.reduce_outliers(
            texts, 
            topics, 
            strategy="embeddings",
            threshold=0.1
        )
        topic_model._topic_model.update_topics(texts, topics=new_topics)
        topic_info = parse_topic_info(self._topic_model.get_topic_info())
        return topic_info, new_topics

    def __del__(self):
        """Automatically free GPU memory when the object is deleted."""
        self.embedder = None
        torch.cuda.empty_cache()

In [ ]:
topic_model = TopicModel("qwen")

In [ ]:
outdir = "../outputs/opentext_v3/"
os.makedirs(outdir, exist_ok=True)

## Q46: Are there reasons why you choose to use a specific language over others?

In [ ]:
# Run the full pipeline on questions
qid = "Q46"
texts = prepare_texts(main_df, qid)
topic_info, topics, hier_topics = topic_model.compute_topics(
    open_text_Qs[qid],
    texts, 
    summarization_model="openrouter/google/gemma-3-27b-it",
    umap_n_neighbors=10,
    umap_n_components=20,
    hdbscan_min_cluster_size=10,
    hdbscan_min_samples=5,
    # hdbscan_cluster_selection_method="eom"
    exclude_stopwords=["no", "nessuno", "nessuna"],
    reuse_embeddings=True
)
topic_info.to_csv(os.path.join(outdir, f"{qid}_topics.tsv"), sep='\t', index=False)
topics_df = pd.DataFrame({"response": texts, "topic": topics}).sort_values("topic")
topics_df.to_csv(os.path.join(outdir, f"{qid}_labels.tsv"), sep="\t", index=False)

In [ ]:
topic_model._topic_model.visualize_hierarchy(hierarchical_topics=hier_topics)

After the hierarchical analysis and manual inspection of assigned labels, we can merge overlapping clusters (esp. the less populated with the most populated ones).

In [ ]:
topics_to_merge = [[0, 7], [3, 6, 8, 12, 13], [9, 10], [11, 14]]
topic_info, topics = topic_model.merge_topics(texts, topics_to_merge)

In [ ]:
topic_info.to_csv(os.path.join(outdir, f"{qid}_topics_merged.tsv"), sep='\t', index=False)
topics_df.to_csv(os.path.join(outdir, f"{qid}_labels_merged.tsv"), sep="\t", index=False)

## Q48: Are there specific reasons why you prefer writing or speaking?

In [ ]:
# Run the full pipeline on questions
qid = "Q48"
texts = prepare_texts(main_df, qid)
topic_info, topics, hier_topics = topic_model.compute_topics(
    open_text_Qs[qid],
    texts, 
    # summarization_model="keybert",
    umap_n_neighbors=5,
    summarization_model="openrouter/google/gemma-3-27b-it",
    umap_n_components=20,
    hdbscan_min_cluster_size=6,
    hdbscan_min_samples=5,
    hdbscan_cluster_selection_method="eom"
    # exclude_stopwords=["no", "nessuno", "nessuna"],
    # reuse_embeddings=True
)
topic_info.to_csv(os.path.join(outdir, f"{qid}_topics.tsv"), sep='\t', index=False)
topics_df = pd.DataFrame({"response": texts, "topic": topics}).sort_values("topic")
topics_df.to_csv(os.path.join(outdir, f"{qid}_labels.tsv"), sep="\t", index=False)

In [ ]:
topic_model._topic_model.visualize_hierarchy(hierarchical_topics=hier_topics)

Merge clusters manually again.

In [ ]:
topics_to_merge = [
    [0, 4, 8, 21],
    [1, 11, 17, 18],
    [2, 6, 10, 13, 14, 20],
    [3, 7],
    [5, 9, 15],
    [16, 22]
    
]
topic_info, topics = topic_model.merge_topics(texts, topics_to_merge)

In [ ]:
topic_info.to_csv(os.path.join(outdir, f"{qid}_topics_merged.tsv"), sep='\t', index=False)
topics_df.to_csv(os.path.join(outdir, f"{qid}_labels_merged.tsv"), sep="\t", index=False)